In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Install dependencies
!pip install -q transformers datasets

In [ ]:
# 3. Imports
import os
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# 4. Paths (update these to your actual paths)
DRIVE_FOLDER = "/content/drive/MyDrive/FIRE-2026"       # <-- update this
RESULTS_PATH = f"{DRIVE_FOLDER}/kg_result.json"                # your uploaded claim+top_3_evidence file
FINAL_MODEL_DIR = "/content/drive/MyDrive/FIRE-2026/FINAL_MODEL_1/final_model"  # <-- your saved trained model
OUTPUT_PATH = f"{DRIVE_FOLDER}/final_predictions_5.json"

In [ ]:
# 5. Label mapping (must match training)
id2label = {0: "SUPPORTS", 1: "REFUTES"}

In [ ]:
# 6. Load tokenizer & trained model
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_DIR)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
# 7. Load input file
with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    input_data = json.load(f)

print(f"Loaded {len(input_data)} claims, each with up to 3 fusion-ranked evidences.")

Loaded 3900 claims, each with up to 3 fusion-ranked evidences.


In [ ]:
# 8. Run inference on every (claim, evidence) pair
output_records = []

with torch.no_grad():
    for entry in input_data:
        claim_id = entry["ID"]
        claim_text = entry["Claim"]
        evidences = entry["top_3_evidence"]

        for idx, ev in enumerate(evidences, start=1):
            evidence_id = ev["ID"]
            evidence_text = ev["Evidence"]
            retrieval_score = ev.get("score", None)

            # Tokenize claim-evidence pair
            inputs = tokenizer(
                claim_text,
                evidence_text,
                truncation=True,
                max_length=192,
                padding="max_length",
                return_tensors="pt",
            ).to(device)

            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)
            pred_label_id = torch.argmax(probs, dim=-1).item()
            confidence = probs[0][pred_label_id].item()

            output_records.append({
                "ID": f"{claim_id}_{idx:02d}",
                "Claim_ID": claim_id,
                "Evidence_ID": evidence_id,
                "Claim": claim_text,
                "Evidence": evidence_text,
                "Retrieval_Score": retrieval_score,
                "Predicted_Label": id2label[pred_label_id],
                "Confidence": round(confidence, 4),
            })

print(f"✅ Total predictions generated: {len(output_records)}")

✅ Total predictions generated: 11700


In [ ]:
# 9. Save to output file
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output_records, f, ensure_ascii=False, indent=2)

print(f"✅ Predictions saved to: {OUTPUT_PATH}")

✅ Predictions saved to: /content/drive/MyDrive/FIRE-2026/final_predictions_5.json


In [ ]:
# 10. Preview first few results
for rec in output_records[:50]:
    print(f"{rec['ID']} | {rec['Predicted_Label']} (conf={rec['Confidence']}) | {rec['Claim'][:50]}...")

S1/T/BN/1001_01 | REFUTES (conf=0.9022) | সুকান্ত মজুমদার গুন্ডামি করতে শুরু করলে..., কেন এক...
S1/T/BN/1001_02 | REFUTES (conf=0.9018) | সুকান্ত মজুমদার গুন্ডামি করতে শুরু করলে..., কেন এক...
S1/T/BN/1001_03 | REFUTES (conf=0.902) | সুকান্ত মজুমদার গুন্ডামি করতে শুরু করলে..., কেন এক...
S1/T/BN/1002_01 | REFUTES (conf=0.9016) | ইউনুসরা ছাত্রদের দাবি উপেক্ষা করে জুলাই বিপ্লব নিয়...
S1/T/BN/1002_02 | REFUTES (conf=0.9021) | ইউনুসরা ছাত্রদের দাবি উপেক্ষা করে জুলাই বিপ্লব নিয়...
S1/T/BN/1002_03 | REFUTES (conf=0.9025) | ইউনুসরা ছাত্রদের দাবি উপেক্ষা করে জুলাই বিপ্লব নিয়...
S1/T/BN/1003_01 | SUPPORTS (conf=0.8433) | ‘করোনাকে কুপোকাত করতে পরিকল্পনা বানান’, সার্ক নেতা...
S1/T/BN/1003_02 | SUPPORTS (conf=0.8439) | ‘করোনাকে কুপোকাত করতে পরিকল্পনা বানান’, সার্ক নেতা...
S1/T/BN/1003_03 | SUPPORTS (conf=0.8456) | ‘করোনাকে কুপোকাত করতে পরিকল্পনা বানান’, সার্ক নেতা...
S1/T/BN/1004_01 | SUPPORTS (conf=0.8472) | মহারাষ্ট্রের মুখ্যমন্ত্রী পদে শপথ দেবেন্দ্রর, সঙ্গ...
S1/T/BN/1004_02 | SUPPORTS (conf=0.84